In [ ]:
import polars as pl
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

daily = pl.read_parquet("../data/1d/1d.parquet").lazy()

Let's play with autogluon

In [ ]:
daily.head().collect(engine="streaming")

In [ ]:
train_df = (
    daily.drop(["ignore", "close_time"])
    .sort(["symbol", "open_time"])
    .with_columns(
        pl.col("close").pct_change().over(pl.col("symbol")).alias("return"),
    )
    .with_columns(pl.col("return").rank().over("open_time").alias("return_rank"))
    .with_columns(pl.col("return_rank") / pl.col("return_rank").max().over("open_time"))
    .drop_nulls()
    .filter(pl.col("open_time") < pl.datetime(2024, 1, 1))
    .collect(engine="streaming")
    .to_pandas()
)

In [ ]:
full_df = (
    daily.drop(["ignore", "close_time"])
    .sort(["symbol", "open_time"])
    .with_columns(
        pl.col("close").pct_change().over(pl.col("symbol")).alias("return"),
    )
    .with_columns(pl.col("return").rank().over("open_time").alias("return_rank"))
    .with_columns(pl.col("return_rank") / pl.col("return_rank").max().over("open_time"))
    .drop_nulls()
    .filter((pl.col("open_time") >= pl.datetime(2024, 1, 1)) & (pl.col("open_time") < pl.datetime(2025, 1, 1)))
    .collect(engine="streaming")
    .to_pandas()
)

full_df = TimeSeriesDataFrame.from_data_frame(full_df, id_column="symbol", timestamp_column="open_time")

In [ ]:
train_df = TimeSeriesDataFrame.from_data_frame(train_df, id_column="symbol", timestamp_column="open_time")
predictor = TimeSeriesPredictor(target="return_rank",freq="1D")
predictor.fit(train_data=train_df)

In [ ]:
predicts = predictor.predict(full_df)

In [ ]:
predictor.predict

In [ ]:
pl.from_pandas(predicts["mean"].reset_index()).with_columns(pl.col("timestamp").cast(pl.Datetime(time_unit='ms')))

In [ ]:
result = (
    daily.drop(["ignore", "close_time"])
    .sort(["symbol", "open_time"])
    .with_columns(
        pl.col("close").pct_change().over(pl.col("symbol")).alias("return"),
    )
    .with_columns(pl.col("return").rank().over("open_time").alias("return_rank"))
    .with_columns(pl.col("return_rank") / pl.col("return_rank").max().over("open_time"))
    .drop_nulls()
    .filter(pl.col("open_time") == pl.datetime(2024, 1, 1))
    .select(
        pl.col("symbol").alias("item_id"),
        pl.col("open_time").alias("timestamp"),
        pl.col("return"),
    )
    .collect(engine="streaming")
    .join(
        pl.from_pandas(predicts["mean"].reset_index()).with_columns(
            pl.col("timestamp").cast(pl.Datetime(time_unit="ms"))
        ),
        on=["item_id", "timestamp"],
    )
)

Let's check the average return of a volume-rocketing event.

In [ ]:
result.select(pl.col("return").rank(), pl.col("mean").rank()).corr()

In [ ]:
daily.with_columns(pl.col("close_time").cast(pl.Datetime)).with_columns(
    pl.col("close").pct_change().over("symbol").alias("return")
).group_by("close_time").agg(pl.col("return").mean()).sort("close_time").with_columns(
    pl.col("return").cum_sum().alias("cumulative_return")).collect(engine="streaming").plot.line(x="close_time", y="cumulative_return")

In [ ]:
data = daily.sort("close_time").with_columns(
    (pl.col("close").pct_change().shift(-1)).over("symbol").alias("return")
)
data.with_columns(
    ((pl.col("volume") / pl.col("volume").shift(1)).over("symbol")).alias(
        "volume_change"
    )
).filter(pl.col("volume_change") > 5.0).select(
    pl.col("return"), pl.col("close_time")
).group_by("close_time").agg(pl.col("return").mean()).with_columns(
    pl.col("return").cum_sum().alias("cum_return")
).sort("close_time").collect(engine="streaming").plot.line(
    x="close_time", y="cum_return"
)

In [ ]:
data = daily.with_columns(pl.col("close").pct_change().over("symbol").alias("return"))
data.with_columns(
    ((pl.col("volume") / pl.col("volume").shift(1)).over("symbol")).alias(
        "volume_change"
    )
).filter(pl.col("volume_change") > 5.0).count().collect(engine="streaming")

In [ ]:
data.select(pl.col("close_time")).unique().count().collect(engine="streaming")

In [ ]:
data = daily.with_columns(pl.col("close").pct_change().over("symbol").alias("return"))
data.with_columns(
    ((pl.col("volume") / pl.col("volume").shift(1)).over("symbol")).alias(
        "volume_change"
    )
).filter(pl.col("volume_change") > 5.0).select(pl.col("close_time")).unique().count().collect(engine="streaming")